# Data preprocessing

## 1. Configuration

### 1.1. CONSTANTS and Environment setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from varname import nameof
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
ROOT = Path().cwd().resolve().parent
SRC = ROOT.parent / "datasets"
OUTPUT = ROOT.parent / "outputs"
OUTPUT_VARIANTS = {
                    "EDA": OUTPUT / "EDA",
                    "data_cleaning": OUTPUT / "data_cleaning",
                    "data_visualization": OUTPUT / "data_visualization",
                    "hypothesis_testing": OUTPUT / "hypothesis_testing",
                    "threshold_finetuning": OUTPUT / "threshold_finetuning"
                  }
print(f"root directory: {ROOT}")
print(f"src directory: {SRC}")
print(f"output directory: {OUTPUT}")

root directory: /home/annoy/projects/team/capstone-1/dev/data_analysis/src
src directory: /home/annoy/projects/team/capstone-1/dev/data_analysis/datasets
output directory: /home/annoy/projects/team/capstone-1/dev/data_analysis/outputs


In [3]:
OUTPUT.mkdir(parents = True, exist_ok=True)
print(f"{OUTPUT} is created.") if (OUTPUT.exists() and OUTPUT.is_dir()) else print(f"{OUTOUT} is not created")
for directory in OUTPUT_VARIANTS.values():
    (directory).mkdir(parents=True, exist_ok= True)
    if (directory).exists() and (directory).is_dir():
        print(f"{directory} is created.")


/home/annoy/projects/team/capstone-1/dev/data_analysis/outputs is created.
/home/annoy/projects/team/capstone-1/dev/data_analysis/outputs/EDA is created.
/home/annoy/projects/team/capstone-1/dev/data_analysis/outputs/data_cleaning is created.
/home/annoy/projects/team/capstone-1/dev/data_analysis/outputs/data_visualization is created.
/home/annoy/projects/team/capstone-1/dev/data_analysis/outputs/hypothesis_testing is created.
/home/annoy/projects/team/capstone-1/dev/data_analysis/outputs/threshold_finetuning is created.


### 1.2. Plotting configuration

In [4]:
    # 1. Matplotlib Global Configuration (rcParams)
plt.rcParams.update({
    # Figure dimensions and resolution
    "figure.figsize": (8, 5),          # Standard aspect ratio for reports and notebooks
    "figure.dpi": 100,                 # Display resolution
    "savefig.dpi": 300,                # High-resolution export for publication
    "savefig.bbox": "tight",           # Prevent clipping of axis labels and titles
    
    # Typography
    "font.family": "sans-serif",       # Clean, readable default font family
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
    "font.size": 11,                   # Base font size for readability
    "axes.titlesize": 13,              # Subplot title font size
    "axes.labelsize": 11,              # X and Y axis label font size
    "xtick.labelsize": 10,             # Tick label size
    "ytick.labelsize": 10,
    
    # Axes and Grid Styling
    "axes.grid": True,                 # Enable background grid for precise data tracking
    "grid.alpha": 0.3,                 # Subtle grid transparency to avoid overpowering data
    "grid.linestyle": "--",            # Dashed grid lines
    "axes.linewidth": 0.8,             # Spine line width
    "axes.edgecolor": "#333333",       # Dark gray spines instead of harsh pure black
    
    # Lines and Markers
    "lines.linewidth": 1.5,            # Default line thickness for plots
    "patch.linewidth": 0.5,            # Border width for histograms, bars, and KDE fills
    
    # Layout Management
    "figure.autolayout": True          # Automatic tight layout padding adjustment
})
print(f"Loaded Mathplotlib Configuration")

sns.set_theme(style="whitegrid", palette="deep", font="sans-serif")
print(f"Loaded Seaborn Configuration")


Loaded Mathplotlib Configuration
Loaded Seaborn Configuration


### 1.3. Functions

In [17]:
def load_files(src = SRC):
    suspicious_account_data = pd.read_csv(src/"accounts.csv")
    suspicious_transactions_data = pd.read_csv(SRC / 'transactions.csv')
    suspicious_blacklisted_data = pd.read_csv(src/"blacklisted_wallets.csv")
    suspicious_wallet_hops_data = pd.read_csv(src/"wallet_hops.csv")
    return [suspicious_account_data, suspicious_transactions_data, suspicious_blacklisted_data, suspicious_wallet_hops_data]
    
all_dfs = load_files() 
suspicious_account_data, suspicious_transactions_data, suspicious_blacklisted_data, suspicious_wallet_hops_data = all_dfs
all_dfs_name = [nameof(suspicious_account_data),
                nameof(suspicious_wallet_hops_data), 
                nameof(suspicious_transactions_data),
                nameof(suspicious_blacklisted_data)]
# create a copy for cleaning. 
cleaned_suspicious_account_data = suspicious_account_data.copy(deep=True)
cleaned_suspicious_transactions_data = suspicious_transactions_data.copy(deep=True)
cleaned_suspicious_transactions_data = suspicious_blacklisted_data.copy(deep=True)
cleaned_suspicious_wallet_hops_data = suspicious_wallet_hops_data.copy(True)

In [14]:
suspicious_account_data.sample(5)

,address,hex_address,first_seen_timestamp,last_seen_timestamp,created_at
386771,TLHLi1BLKxZ9KKpW199dMtnM3MpUnPLdRT,NaN,NaN,NaN,2026-10-29 04:17:16.536448+00
377748,TWDBqLXKFq4Yqc42pVLYzjVeF7oYDbwRFH,NaN,NaN,NaN,2026-10-29 04:03:37.053444+00
423582,TSejSotBSx6mUZtTzZ6D16SPEygBUD8BpQ,NaN,NaN,NaN,2026-10-29 05:04:51.495593+00
655965,TNRxbQm1tZ1CEdvtcQSMGV2vewTWsYTaB5,NaN,NaN,NaN,2026-10-29 11:50:58.680465+00
650369,TFRc3uhqCVkBNGRLrvM6PjTfNnah3kTZWy,NaN,2024-02-17 05:31:06+00,2024-02-17 05:31:06+00,2026-10-29 11:46:54.309977+00


## 2. Drop irrelevant columns

In [8]:
cleaned_suspicious_wallet_hops_data = cleaned_suspicious_wallet_hops_data.drop(columns=["parent_address","via_tx_hash"])
suspicious_wallet_hops_data.columns

Index(['hop_id', 'wallet_address', 'seed_address', 'hop_level', 'direction',
       'discovered_at'],
      dtype='str')

## 3. Drop NA Values

In [7]:
## Drop the contract address 
cleaned_suspicious_transactions_data = cleaned_suspicious_transactions_data.drop(columns=["contract_address"])
suspicious_transactions_data.columns

Index(['tx_hash', 'block_number', 'block_timestamp', 'from_address',
       'to_address', 'token_symbol', 'amount', 'fee_trx', 'created_at'],
      dtype='str')

## 3. Statistical Analysis

In [9]:
import scipy.stats as stats

In [10]:
cleaned_suspicious_account_data.select_dtypes(include=["object"]).columns

Index(['address', 'hex_address', 'first_seen_timestamp', 'last_seen_timestamp',
       'created_at'],
      dtype='str')

In [11]:
suspicious_wallet_hops_data.describe().mean()

hop_id       39322.115470
hop_level     9117.036857
dtype: float64

In [12]:
# CV for all dfs
df_lst_
def cv_df(all_dfs_name, all_df):
    for df in all_dfs: 
        

_IncompleteInputError: incomplete input (690523781.py, line 5)

In [ ]:
suspicious_wallet_hops_data.info()